# Fooocus_extend – VSW Google Colab

Robuste Schulungsfassung mit **isolierter Python-Umgebung**, Fortschrittsanzeige, direkter Civitai-Modellwahl und Runtime-Diagnose.

**Stabilitätsprofil für Schulungen:** Cloudflared ist der bevorzugte öffentliche Tunnel. Gradio Share bleibt als Fallback verfügbar. Auf T4/16-GB-GPUs ist der zusätzliche High-VRAM-Modus standardmäßig deaktiviert.

**Start:** Einstellungen wählen und auf die Start-Zelle klicken. Sobald `[100%] WEBOBERFLÄCHE BEREIT` erscheint, den öffentlichen Link öffnen.

Bei `Reconnect`, `Unexpected token '<'` oder einem abgebrochenen Generate-Lauf anschließend die Zelle **🩺 VSW-DIAGNOSE** ausführen. Sie trennt Backend-, Tunnel- und GPU-/VRAM-Probleme.

Empfohlen: **Runtime 2025.07 · Python 3.11 · GPU/T4**.


In [ ]:

# @title ▶ Fooocus_extend STARTEN / NEUSTARTEN
import os, urllib.request

Fooocus_Profile = "realistic" #@param ["default", "realistic", "anime"]
Fooocus_Theme = "dark" #@param ["dark", "light"]
Tunnel = "cloudflared" #@param ["cloudflared", "gradio"]
Memory_patch = False #@param {type:"boolean"}
GoogleDrive_output = False #@param {type:"boolean"}
Use_latest_main = False #@param {type:"boolean"}
Force_rebuild_environment = False #@param {type:"boolean"}

# Optional: Civitai-Version-IDs oder direkte Download-URLs, mehrere Einträge mit ; trennen.
Civitai_Checkpoints = "" #@param {type:"string"}
Civitai_LoRAs = "" #@param {type:"string"}
Use_Civitai_Secret = False #@param {type:"boolean"}

if Use_Civitai_Secret:
    try:
        from google.colab import userdata
        token = userdata.get('CIVITAI_TOKEN')
        if token:
            os.environ['CIVITAI_TOKEN'] = token
            print('Civitai-Token aus Colab Secrets geladen.')
    except Exception as e:
        raise RuntimeError("Colab Secret 'CIVITAI_TOKEN' konnte nicht geladen werden.") from e

# Teststand dieses Arbeitsbranches. Vor Merge auf main wird dieser Wert auf 'main' gesetzt.
VSW_Code_Ref = "work/runtime-health-cloudflared"
os.environ['VSW_HELPER_REF'] = VSW_Code_Ref
url = f"https://raw.githubusercontent.com/MlIelAst/Fooocus_Extend_VSW_Colab/{VSW_Code_Ref}/vsw_launcher.py"
print(f"VSW-Launcher wird geladen – Ref: {VSW_Code_Ref}")
script = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

replacements = {
    'PROFILE = "realistic"': f'PROFILE = {Fooocus_Profile!r}',
    'THEME = "dark"': f'THEME = {Fooocus_Theme!r}',
    'TUNNEL = "cloudflared"': f'TUNNEL = {Tunnel!r}',
    'MEMORY_PATCH = False': f'MEMORY_PATCH = {Memory_patch!r}',
    'GOOGLE_DRIVE_OUTPUT = False': f'GOOGLE_DRIVE_OUTPUT = {GoogleDrive_output!r}',
    'USE_LATEST_MAIN = False': f'USE_LATEST_MAIN = {Use_latest_main!r}',
    'FORCE_REBUILD = False': f'FORCE_REBUILD = {Force_rebuild_environment!r}',
    'EXTRA_CHECKPOINTS = ""': f'EXTRA_CHECKPOINTS = {Civitai_Checkpoints!r}',
    'EXTRA_LORAS = ""': f'EXTRA_LORAS = {Civitai_LoRAs!r}',
}
for old, new in replacements.items():
    if old not in script:
        raise RuntimeError(f"VSW-Launcher unerwartet geändert: Einstellung fehlt: {old}")
    script = script.replace(old, new, 1)

exec(compile(script, "vsw_launcher.py", "exec"), {"__name__": "__main__"})


In [ ]:

# @title 🩺 VSW-DIAGNOSE: Backend · Tunnel · GPU prüfen
import os, sys, urllib.request
from pathlib import Path

VSW_Code_Ref = "work/runtime-health-cloudflared"
helper = Path('/content/vsw_runtime.py')
url = f"https://raw.githubusercontent.com/MlIelAst/Fooocus_Extend_VSW_Colab/{VSW_Code_Ref}/vsw_runtime.py"
urllib.request.urlretrieve(url, helper)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import importlib
import vsw_runtime
importlib.reload(vsw_runtime)
vsw_runtime.print_runtime_diagnosis('/content/fooocus_extend_startup.log')


## Empfohlene VSW-Einstellungen

- `Tunnel = cloudflared` – bevorzugt für Schulungen; `gradio` nur als Fallback.
- `Memory_patch = False` – für T4/16 GB stabiler, insbesondere bei FaceEnhancer/ADetailer und mehreren Bildern.
- `Use_latest_main = False` – reproduzierbaren Fooocus_extend-Stand verwenden.
- `GoogleDrive_output = False` – datensparsamer Standard.

## Wenn die Oberfläche `Reconnect` zeigt

1. Nicht mehrfach auf Reconnect klicken.
2. Die Zelle **🩺 VSW-DIAGNOSE** ausführen.
3. Zeigt sie `lokal OK / öffentlich FEHLER`, ist primär der Tunnel betroffen.
4. Zeigt sie `Fooocus-Prozess OFFLINE / lokal FEHLER`, das Log auf `CUDA out of memory`, `Killed`, `Traceback` oder Zusatzmodulfehler prüfen.
5. Für den ersten Stabilitätstest: 1 Bild, ADetailer/FaceEnhancer aus; anschließend Funktionen schrittweise zuschalten.

## Modelle und LoRAs

Für Fooocus bevorzugt **SDXL-Checkpoints** und dazu passende **SDXL-LoRAs** verwenden. Checkpoints landen unter `/content/Fooocus_extend/models/checkpoints`, LoRAs unter `/content/Fooocus_extend/models/loras`. Mehrere Civitai-Version-IDs oder direkte Download-URLs mit `;` trennen. Für geschützte Downloads `CIVITAI_TOKEN` als Colab Secret hinterlegen.

Vollständiges Serverlog: `/content/fooocus_extend_startup.log`. Browser-Erweiterungsmeldungen wie `Sentry.init` oder `content.js` sind nicht automatisch Fooocus-Kernfehler.

Nach der Übung: **Laufzeit → Laufzeit trennen und löschen**.
